# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: TÙY CHỌN khi thực thi mã C++ hoặc Rust</h2>
            <span style="color:#f71;">Cách khác: bạn có thể chạy trên website đã giới thiệu hôm qua</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model cao cấp GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4 — những model có giá hơi cao hơn. Chi phí vẫn thấp, nhưng nếu bạn muốn giữ chi phí cực thấp, hãy chọn model rẻ hơn như gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Nạp các thư viện phục vụ cấu hình API, gọi model, chạy trình biên dịch và dựng giao diện.

import os  # Đọc các biến môi trường chứa API key.
import io  # Tạo bộ đệm văn bản để thu lại kết quả do print() sinh ra.
import sys  # Truy cập và tạm thời thay đổi luồng xuất chuẩn sys.stdout.
from dotenv import load_dotenv  # Nạp các biến cấu hình từ file .env.
from openai import OpenAI  # Tạo client tương thích OpenAI để gọi nhiều nhà cung cấp model.
import gradio as gr  # Xây dựng giao diện web tương tác cho ứng dụng chuyển mã.
import subprocess  # Gọi trình biên dịch và chạy chương trình hệ thống từ Python.
from IPython.display import Markdown, display  # Hiển thị phản hồi Markdown đẹp trong notebook.


In [ ]:
load_dotenv(override=True)  # Đọc file .env và cho phép giá trị trong file ghi đè biến môi trường hiện có.
openai_api_key = os.getenv('OPENAI_API_KEY')  # Lấy khóa dùng để gọi API OpenAI.
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')  # Lấy khóa Anthropic/Claude nếu đã cấu hình.
google_api_key = os.getenv('GOOGLE_API_KEY')  # Lấy khóa Google/Gemini nếu đã cấu hình.
grok_api_key = os.getenv('GROK_API_KEY')  # Lấy khóa xAI/Grok nếu đã cấu hình.
groq_api_key = os.getenv('GROQ_API_KEY')  # Lấy khóa Groq nếu đã cấu hình.
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')  # Lấy khóa cổng model OpenRouter nếu đã cấu hình.

if openai_api_key:  # Kiểm tra khóa OpenAI có tồn tại hay không.
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")  # In 8 ký tự đầu để xác nhận mà không làm lộ toàn bộ khóa.
else:  # Chạy nhánh này khi không tìm thấy khóa OpenAI.
    print("Chưa thiết lập OpenAI API Key")  # Thông báo OpenAI chưa được cấu hình.
    
if anthropic_api_key:  # Kiểm tra khóa Anthropic có tồn tại hay không.
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")  # Chỉ in 7 ký tự đầu của khóa để kiểm tra.
else:  # Chạy khi chưa có khóa Anthropic.
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")  # Nhắc rằng khóa Claude là tùy chọn.

if google_api_key:  # Kiểm tra khóa Google có tồn tại hay không.
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")  # Chỉ in 2 ký tự đầu để nhận biết khóa.
else:  # Chạy khi chưa có khóa Google.
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")  # Nhắc rằng Gemini không bắt buộc cho mọi lựa chọn model.

if grok_api_key:  # Kiểm tra khóa Grok có tồn tại hay không.
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")  # Chỉ in 4 ký tự đầu của khóa Grok.
else:  # Chạy khi chưa có khóa Grok.
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")  # Báo rằng Grok chưa được cấu hình.

if groq_api_key:  # Kiểm tra khóa Groq có tồn tại hay không.
    print(f"Groq API Key tồn tại và bắt đầu bằng {groq_api_key[:4]}")  # Chỉ in phần đầu khóa để xác nhận.
else:  # Chạy khi chưa có khóa Groq.
    print("Chưa thiết lập Groq API Key (và đây là tùy chọn)")  # Báo rằng Groq chưa được cấu hình.

if openrouter_api_key:  # Kiểm tra khóa OpenRouter có tồn tại hay không.
    print(f"OpenRouter API Key tồn tại và bắt đầu bằng {openrouter_api_key[:6]}")  # Chỉ in 6 ký tự đầu để xác nhận.
else:  # Chạy khi chưa có khóa OpenRouter.
    print("Chưa thiết lập OpenRouter API Key (và đây là tùy chọn)")  # Báo rằng OpenRouter chưa được cấu hình.



In [ ]:
# Khởi tạo các client API; tất cả dùng giao diện OpenAI nhưng trỏ tới endpoint khác nhau.

openai = OpenAI()  # Client OpenAI mặc định, tự đọc OPENAI_API_KEY từ môi trường.

anthropic_url = "https://api.anthropic.com/v1/"  # Endpoint tương thích OpenAI của Anthropic.
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Endpoint tương thích OpenAI của Gemini.
grok_url = "https://api.x.ai/v1"  # Endpoint API của xAI/Grok.
groq_url = "https://api.groq.com/openai/v1"  # Endpoint tương thích OpenAI của Groq.
ollama_url = "http://localhost:11434/v1"  # Endpoint Ollama chạy model ngay trên máy cục bộ.
openrouter_url = "https://openrouter.ai/api/v1"  # Endpoint OpenRouter để truy cập nhiều model.

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)  # Client dùng khóa Anthropic và endpoint Claude.
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)  # Client dùng khóa Google và endpoint Gemini.
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)  # Client dùng khóa xAI và endpoint Grok.
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)  # Client dùng khóa Groq và endpoint Groq.
ollama = OpenAI(api_key="ollama", base_url=ollama_url)  # Client local; khóa giả chỉ để SDK chấp nhận cấu hình.
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)  # Client dùng khóa và endpoint OpenRouter.



In [ ]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]  # Danh sách model được hiển thị trong dropdown Gradio.

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}  # Ánh xạ mỗi tên model tới đúng client/nhà cung cấp.

# Muốn giữ chi phí cực thấp? Thay bằng các model bạn chọn, dùng ví dụ từ hôm qua

In [ ]:
from system_info import retrieve_system_info, rust_toolchain_info  # Nhập hàm lấy cấu hình máy và kiểm tra Rust toolchain.

system_info = retrieve_system_info()  # Lấy hệ điều hành, CPU và môi trường để tối ưu mã sinh ra.
rust_info = rust_toolchain_info()  # Kiểm tra rustc/cargo và bộ công cụ Rust trên máy.
rust_info  # Hiển thị kết quả kiểm tra Rust trong output của ô notebook.

In [ ]:
# Tạo prompt nhờ model đánh giá máy hiện tại và đề xuất lệnh biên dịch/chạy Rust tối ưu.
message = f"""
Đây là báo cáo thông tin hệ thống (system information) của máy tính tôi.
Tôi muốn chạy Rust compiler (trình biên dịch Rust) để biên dịch một file rust tên main.rs rồi thực thi theo cách đơn giản nhất.
Hãy trả lời xem tôi có cần cài Rust toolchain (bộ công cụ Rust) không. Nếu có, hãy đưa hướng dẫn từng bước đơn giản nhất.

Nếu máy tôi đã sẵn sàng biên dịch Rust, tôi muốn chạy đoạn Python tương tự như sau để biên dịch và thực thi:
```python
compile_command = # điền lệnh ở đây — để đạt runtime performance (hiệu năng khi chạy) nhanh nhất có thể
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)  # Chạy compiler và giữ lại kết quả tiến trình.
run_command = # điền lệnh ở đây
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)  # Chạy binary và thu output.
return run_result.stdout  # Trả stdout của chương trình đã biên dịch.
```
Hãy cho tôi chính xác nên dùng gì cho compile_command và run_command.
Ưu tiên runtime performance tối đa; compile time (thời gian biên dịch) có thể chậm. Điều quan trọng là chạy nhanh nhất trên platform (nền tảng) này.
Trả lời các lệnh bằng markdown.

Thông tin hệ thống:
{system_info}

Thông tin Rust toolchain:
{rust_info}
"""  # Kết thúc prompt kiểm tra và thiết lập Rust.

response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])  # Gửi prompt bằng model đầu tiên trong danh sách và nhận phản hồi.
display(Markdown(response.choices[0].message.content))  # Lấy câu trả lời đầu tiên rồi render dưới dạng Markdown.

## Với C++, ghi đè bằng lệnh từ hôm qua; với Rust, dùng các lệnh mới

Hoặc dùng website như hôm qua:

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
compile_command = [  # Danh sách đối số truyền cho subprocess để biên dịch Rust.
    "/Users/ed/.cargo/bin/rustc",  # Đường dẫn tuyệt đối tới trình biên dịch rustc trên máy tác giả.
    "main.rs",  # File mã nguồn Rust cần biên dịch.
    "-C", "opt-level=3",  # Bật mức tối ưu hóa mã máy cao.
    "-C", "target-cpu=native",  # Tận dụng tập lệnh CPU của máy đang chạy.
    "-C", "codegen-units=1",  # Dùng một đơn vị sinh mã để tối ưu xuyên suốt tốt hơn.
    "-C", "lto=fat",  # Bật tối ưu hóa liên kết toàn phần để cải thiện runtime.
    "-C", "panic=abort",  # Dừng ngay khi panic thay vì tạo cơ chế unwind.
    "-C", "strip=symbols",  # Loại bỏ symbol không cần thiết khỏi file thực thi.
    "-o", "main",  # Đặt tên file thực thi đầu ra là main.
]  # Kết thúc danh sách tham số biên dịch.

run_command = ["./main"]  # Lệnh chạy file thực thi vừa được rustc tạo ra.


## Tiếp theo: nhiệm vụ chính

In [ ]:
language = "Rust"  # Chọn ngôn ngữ đích; có thể đổi thành "C++".
extension = "rs" if language == "Rust" else "cpp"  # Suy ra phần mở rộng file từ ngôn ngữ đã chọn.

# System prompt quy định vai trò, định dạng phản hồi và yêu cầu tương đương output.
system_prompt = f"""
Nhiệm vụ của bạn là chuyển mã Python thành mã {language} hiệu năng cao (high performance).
Chỉ trả lời bằng mã {language}. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã {language} phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""  # Kết thúc chuỗi system prompt.

def user_prompt_for(python):  # Tạo prompt chi tiết cho từng đoạn mã Python cần chuyển đổi.
    # Trả về f-string để chèn ngôn ngữ, cấu hình máy, lệnh compiler và mã nguồn.
    return f"""
Port (chuyển) mã Python này sang {language} với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.{language} rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã {language}.
Mã Python cần port:

```python
{python}
```
"""  # Kết thúc và trả chuỗi user prompt.

In [ ]:
def messages_for(python):  # Đóng gói prompt theo định dạng message mà Chat Completions yêu cầu.
    return [  # Trả về cuộc hội thoại gồm chỉ dẫn hệ thống và yêu cầu người dùng.
        {"role": "system", "content": system_prompt},  # Message hệ thống đặt quy tắc chuyển mã.
        {"role": "user", "content": user_prompt_for(python)}  # Message người dùng chứa mã Python cụ thể.
    ]  # Kết thúc danh sách messages.
 

In [ ]:
def write_output(code):  # Ghi mã do model sinh ra vào file nguồn có đúng phần mở rộng.
    with open(f"main.{extension}", "w") as f:  # Mở main.rs/main.cpp ở chế độ ghi và tự đóng file.
        f.write(code)  # Ghi toàn bộ chuỗi mã nguồn vào file.

In [ ]:
def port(model, python):  # Gửi mã Python tới model được chọn và nhận phiên bản Rust/C++.
    client = clients[model]  # Tra client tương ứng với tên model trong từ điển clients.
    reasoning_effort = "high" if 'gpt' in model else None  # Yêu cầu suy luận cao cho GPT; model khác dùng mặc định.
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)  # Gọi API sinh mã.
    reply = response.choices[0].message.content  # Lấy nội dung phản hồi của lựa chọn đầu tiên.
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')  # Bỏ hàng rào Markdown để thu mã nguồn thuần.
    return reply  # Trả mã đã làm sạch cho giao diện Gradio.

In [ ]:
def run_python(code):  # Thực thi chuỗi mã Python và trả về nội dung mà mã đó in ra.
    globals_dict = {"__builtins__": __builtins__}  # Tạo không gian global riêng nhưng vẫn cho dùng hàm dựng sẵn.

    buffer = io.StringIO()  # Tạo bộ đệm trong RAM để hứng dữ liệu xuất chuẩn.
    old_stdout = sys.stdout  # Lưu stdout gốc để khôi phục sau khi chạy.
    sys.stdout = buffer  # Chuyển mọi print() tạm thời vào bộ đệm.

    try:  # Thử chạy mã và thu kết quả, đồng thời cho phép xử lý lỗi.
        exec(code, globals_dict)  # Thực thi chuỗi code trong không gian global riêng.
        output = buffer.getvalue()  # Đọc toàn bộ nội dung đã được print vào bộ đệm.
    except Exception as e:  # Bắt mọi lỗi Python phát sinh khi thực thi đoạn mã.
        output = f"Lỗi (Error): {e}"  # Chuyển lỗi thành chuỗi để hiển thị trên giao diện.
    finally:  # Khối này luôn chạy dù thành công hay lỗi.
        sys.stdout = old_stdout  # Khôi phục stdout để notebook tiếp tục in bình thường.

    return output  # Trả output hoặc thông báo lỗi về ô kết quả Python.

In [ ]:
# Dùng các lệnh biên dịch/chạy đã được GPT-5 đề xuất ở ô phía trên.

def compile_and_run(code):  # Lưu, biên dịch rồi chạy mã Rust/C++ do model tạo.
    write_output(code)  # Ghi mã nguồn vào main.rs hoặc main.cpp trước khi biên dịch.
    try:  # Thử biên dịch và thực thi để có thể xử lý lỗi tiến trình.
        subprocess.run(compile_command, check=True, text=True, capture_output=True)  # Biên dịch; check=True báo lỗi nếu compiler thất bại.
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)  # Chạy binary và thu stdout/stderr dạng text.
        return run_result.stdout  # Trả output của chương trình khi chạy thành công.
    except subprocess.CalledProcessError as e:  # Bắt lỗi từ bước biên dịch hoặc chạy binary.
        return f"Đã xảy ra lỗi (error):\n{e.stderr}"  # Trả stderr để người dùng biết nguyên nhân lỗi.

In [ ]:
# Lưu bài toán Python mẫu dưới dạng chuỗi để nạp sẵn vào trình soạn thảo Gradio.
python_hard = """# Cẩn thận hỗ trợ số lớn: bài toán dùng nhiều phép tính và có thể tạo tổng lớn.

def lcg(seed, a=1664525, c=1013904223, m=2**32):  # Tạo bộ sinh số giả ngẫu nhiên Linear Congruential Generator.
    value = seed  # Khởi tạo trạng thái hiện tại từ seed đầu vào.
    while True:  # Sinh vô hạn giá trị cho tới khi phía gọi ngừng lấy số.
        value = (a * value + c) % m  # Tính trạng thái LCG tiếp theo bằng công thức truy hồi.
        yield value  # Trả một giá trị rồi tạm dừng generator, giữ lại trạng thái.
        
def max_subarray_sum(n, seed, min_val, max_val):  # Tìm tổng lớn nhất của mọi đoạn con bằng thuật toán vét cạn O(n²).
    lcg_gen = lcg(seed)  # Tạo một generator LCG riêng cho seed của lần chạy này.
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]  # Sinh n số trong đoạn [min_val, max_val].
    max_sum = float('-inf')  # Khởi tạo tổng tốt nhất là âm vô cùng để mọi tổng thực đều có thể thay thế.
    for i in range(n):  # Duyệt từng vị trí có thể làm đầu đoạn con.
        current_sum = 0  # Đặt lại tổng tích lũy cho đầu đoạn i hiện tại.
        for j in range(i, n):  # Mở rộng cuối đoạn từ i tới hết mảng.
            current_sum += random_numbers[j]  # Cộng phần tử mới vào tổng đoạn [i, j].
            if current_sum > max_sum:  # Kiểm tra đoạn vừa mở rộng có tốt hơn kỷ lục hay không.
                max_sum = current_sum  # Cập nhật tổng đoạn con lớn nhất đã gặp.
    return max_sum  # Trả kết quả tốt nhất của một mảng số ngẫu nhiên.

def total_max_subarray_sum(n, initial_seed, min_val, max_val):  # Cộng kết quả max-subarray của 20 bộ dữ liệu.
    total_sum = 0  # Khởi tạo biến tích lũy kết quả của tất cả lần chạy.
    lcg_gen = lcg(initial_seed)  # Dùng LCG cấp ngoài để tạo seed khác nhau nhưng tái lập được.
    for _ in range(20):  # Lặp đúng 20 lần; dấu gạch dưới cho biết không dùng chỉ số vòng lặp.
        seed = next(lcg_gen)  # Lấy seed mới cho bộ dữ liệu tiếp theo.
        total_sum += max_subarray_sum(n, seed, min_val, max_val)  # Tính rồi cộng maximum subarray sum của lần này.
    return total_sum  # Trả tổng kết quả sau 20 lần chạy.

# Tham số (Parameters)
n = 10000  # Số phần tử ngẫu nhiên trong mỗi bộ dữ liệu.
initial_seed = 42  # Seed gốc cố định giúp kết quả có thể tái lập.
min_val = -10  # Cận dưới của số nguyên giả ngẫu nhiên.
max_val = 10  # Cận trên của số nguyên giả ngẫu nhiên.

# Đo thời gian hàm (Timing the function)
import time  # Cung cấp đồng hồ để đo tổng thời gian thực thi.
start_time = time.time()  # Ghi lại mốc thời gian ngay trước khi tính toán.
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)  # Chạy bài toán benchmark và lưu kết quả.
end_time = time.time()  # Ghi lại mốc thời gian ngay sau khi tính xong.

print("Tổng Maximum Subarray Sum (20 lần chạy):", result)  # In kết quả để đối chiếu với mã Rust/C++.
print("Thời gian thực thi (Execution Time): {:.6f} giây".format(end_time - start_time))  # In thời lượng với 6 chữ số thập phân.
"""  # Kết thúc chuỗi chương trình Python benchmark.

In [ ]:
from styles import CSS  # Nhập chuỗi CSS tùy chỉnh để tạo kiểu cho giao diện.

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port (chuyển) từ Python sang {language}") as ui:  # Tạo ứng dụng Gradio gốc với CSS, theme và tiêu đề.
    with gr.Row(equal_height=True):  # Xếp hai trình soạn thảo mã trên cùng một hàng, cao bằng nhau.
        with gr.Column(scale=6):  # Cột trái chiếm một nửa hàng.
            python = gr.Code(  # Tạo trình soạn thảo cho mã Python đầu vào.
                label="Python (bản gốc)",  # Nhãn mô tả nội dung của cột trái.
                value=python_hard,  # Nạp sẵn bài toán benchmark vào editor.
                language="python",  # Bật tô sáng cú pháp Python.
                lines=26  # Đặt chiều cao hiển thị tương đương 26 dòng.
            )  # Hoàn tất cấu hình editor Python.
        with gr.Column(scale=6):  # Cột phải chiếm nửa còn lại của hàng.
            cpp = gr.Code(  # Tạo trình soạn thảo nhận mã đích do model sinh.
                label=f"{language} (do model sinh)",  # Nhãn thay đổi theo ngôn ngữ đích.
                value="",  # Khởi tạo editor kết quả bằng chuỗi rỗng.
                language="cpp",  # Dùng chế độ tô sáng C++; cần đổi nếu muốn tô sáng Rust chuẩn.
                lines=26  # Cho editor đầu ra cùng chiều cao với editor Python.
            )  # Hoàn tất cấu hình editor mã đích.

    with gr.Row(elem_classes=["controls"]):  # Tạo hàng điều khiển và gắn class CSS controls.
        python_run = gr.Button("Chạy Python", elem_classes=["run-btn", "py"])  # Nút chạy mã Python gốc.
        model = gr.Dropdown(models, value=models[0], show_label=False)  # Menu chọn model, mặc định phần tử đầu tiên.
        convert = gr.Button(f"Port sang {language}", elem_classes=["convert-btn"])  # Nút yêu cầu model chuyển mã.
        cpp_run = gr.Button(f"Chạy {language}", elem_classes=["run-btn", "cpp"])  # Nút biên dịch và chạy mã đích.

    with gr.Row(equal_height=True):  # Tạo hàng hai ô output có chiều cao bằng nhau.
        with gr.Column(scale=6):  # Cột output bên trái dành cho Python.
            python_out = gr.TextArea(label="Kết quả Python", lines=8, elem_classes=["py-out"])  # Hiển thị stdout/lỗi Python.
        with gr.Column(scale=6):  # Cột output bên phải dành cho ngôn ngữ đích.
            cpp_out = gr.TextArea(label=f"Kết quả {language}", lines=8, elem_classes=["cpp-out"])  # Hiển thị stdout/lỗi Rust/C++.

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])  # Khi bấm Port, gọi model và đổ mã sinh ra vào editor phải.
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])  # Khi bấm Chạy Python, thực thi editor trái và hiện kết quả.
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])  # Khi bấm Chạy ngôn ngữ đích, biên dịch/chạy editor phải.

ui.launch(inbrowser=True)  # Khởi động server Gradio và tự mở giao diện trong trình duyệt.


## KẾT QUẢ!

Qwen 2.5 Coder: FAIL (thất bại)  
Gemini 2.5 Pro: FAIL (thất bại)  
DeepSeek Coder v2: FAIL (thất bại)  
Qwen3 Coder 30B: FAIL (thất bại)  
Claude Sonnet 4.5: FAIL (thất bại)    
GPT-5: FAIL (thất bại)    

Hạng 3: GPT-oss-20B: 0.000341  
Hạng 2: Grok 4: 0.000317  
**Hạng 1: OpenAI GPT-OSS 120B: 0.000304**  

In [ ]:
print(f"Trong thí nghiệm của Ed, kết quả model GPT-OSS 120B nhanh hơn mã Python {33.755209/0.000304:,.0f} lần.")  # Chia hai thời gian benchmark, định dạng không có phần thập phân rồi in hệ số tăng tốc.